In [2]:
import os

# உங்கள் டேட்டாசெட் ஃபোল்டர் பாதையைக் குறிப்பிடவும்
DATASET_PATH = "dataset_binary"

total_images = 0

print("--- Dataset Image Count ---")

# ஒவ்வொரு உட்பிரிவு ஃபোল்டரையும் சரிபார்த்தல்
if os.path.exists(DATASET_PATH):
    for folder in os.listdir(DATASET_PATH):
        folder_path = os.path.join(DATASET_PATH, folder)

        if os.path.isdir(folder_path):
            # ஃபোল்டரில் உள்ள படங்களை மட்டும் எண்ணுதல் (.jpg, .jpeg, .png, .webp)
            images = [
                f
                for f in os.listdir(folder_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
            ]
            count = len(images)
            total_images += count
            print(f"{folder}: {count} images")

    print("---------------------------")
    print(f"Total Images in Dataset: {total_images}")
else:
    print(f"Error: Path '{DATASET_PATH}' does not exist!")

--- Dataset Image Count ---
non_soil: 250 images
soil: 1015 images
---------------------------
Total Images in Dataset: 1265


In [3]:
pip install bing-image-downloader


[notice] A new release of pip is available: 24.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import shutil
from bing_image_downloader import downloader

# 1. Target Directory Setup
TARGET_DIR = os.path.join("dataset_binary", "non_soil")
os.makedirs(TARGET_DIR, exist_ok=True)

# 2. Count current images and calculate remaining needed images
existing_files = [
    f
    for f in os.listdir(TARGET_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]
current_count = len(existing_files)
target_total = 1015
needed_count = target_total - current_count

print(f"Current Non-Soil Images: {current_count}")
print(f"Target Non-Soil Images: {target_total}")

if needed_count <= 0:
    print("You already have enough non-soil images!")
else:
    print(f"Downloading {needed_count} new images from the web...\n")

    # 3. Search Categories & Proportional Download Counts
    # 765 படங்களை 4 பிரிவுகளாகப் பிரித்து பதிவிறக்குதல்
    queries = {
        "cars and vehicles on road": 195,
        "modern city buildings architecture": 190,
        "dogs and pets portraits": 190,
        "people standing group photo": 190,
    }

    temp_dir = "temp_downloads"

    for query, limit in queries.items():
        print(f"Scraping: '{query}' (Target: {limit} images)...")
        downloader.download(
            query,
            limit=limit,
            output_dir=temp_dir,
            adult_filter_off=True,
            force_replace=False,
            timeout=30,
            verbose=False,
        )

        # Move downloaded files to target non_soil folder
        query_folder = os.path.join(temp_dir, query)
        if os.path.exists(query_folder):
            for img_name in os.listdir(query_folder):
                src_path = os.path.join(query_folder, img_name)
                # Create a clean unique filename
                clean_name = f"scraped_{query.replace(' ', '_')}_{img_name}"
                dst_path = os.path.join(TARGET_DIR, clean_name)
                
                # Move image file
                try:
                    shutil.move(src_path, dst_path)
                except Exception as e:
                    pass

    # Clean up temporary downloads directory
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    # 4. Final Count Verification
    final_files = [
        f
        for f in os.listdir(TARGET_DIR)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]
    print("\n-------------------------------------------")
    print("Scraping Completed Successfully!")
    print(f"New Total Non-Soil Images Count: {len(final_files)}")
    print("-------------------------------------------")

Current Non-Soil Images: 250
Target Non-Soil Images: 1015

Scraping: 'cars and vehicles on road' (Target: 195 images)...
[%] Downloading Images to c:\Users\DELL\Downloads\Soil-Type-Classification-master\temp_downloads\cars and vehicles on road
[!] Issue getting: https://media.autoexpress.co.uk/image/private/s--X-WVjvBW--/f_auto,t_content-image-full-desktop@1/v1673264039/autoexpress/2023/01/Best%20new%20cars%202023-23.jpg
[!] Error:: HTTP Error 404: Not Found
[!] Issue getting: https://res.cloudinary.com/total-dealer/image/upload/w_3840,f_auto,q_75/v1/production/8hmxda3xrrzscxli431lucrd6o3x
[!] Error:: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)>
[!] Issue getting: https://www.hdwallpapers.in/download/2024_lamborghini_revuelto_car_4k_5k_hd_cars-5120x2880.jpg
[!] Error:: HTTP Error 500: Internal Server Error
[%] No new images found, stopping.


[%] Done. Downloaded 32 images.
Scraping: 'modern city buildi